# Kaggriculture Agent Development

**Washamba Bots** &mdash; development and evaluation notebook for `nikaangukia_meroni`,
our agent for the Kaggle Kaggriculture simulation competition.

Two agents each manage a farm across a 30-day season (720 turns) and compete for the
higher closing bank balance. There is no static train/test split: every result comes
from running live episodes.

This notebook covers the full working loop we use to develop the agent:

| Section | Purpose |
|---|---|
| 1. Environment setup | Install and pin the simulator, verify the runtime |
| 2. The simulation | Read the authoritative game parameters from the engine |
| 3. Observation schema | Inspect what the agent actually receives each turn |
| 4. Agent under test | Load the current agent from `main.py` |
| 5. Evaluation methodology | Seeded batches, and why single episodes mislead |
| 6. Results | Performance against all three reference opponents |
| 7. Behavioural diagnostics | Where turns go, and what the farm looks like at season end |
| 8. Findings | What the measurements taught us |
| 9. Next steps | Prioritised backlog |

**Prerequisite.** Select the `Python 3.13 (washamba_bots)` kernel. If it is not listed,
register it once from the project root:

```bash
.venv/Scripts/python.exe -m ipykernel install --user \
    --name washamba-bots --display-name "Python 3.13 (washamba_bots)"
```

## 1. Environment setup

The engine is patched mid-competition. A balance change in August 2026 reduced Town
Center demand and made shop unlocks sample *with* replacement, so competition staff
directed everyone to `kaggle-environments >= 1.32.6`. The official starter notebook
still pins `>= 1.32.2`, which predates that patch &mdash; do not copy it.

`%pip` installs into the *running kernel*. `!pip` shells out to whatever `pip` appears
first on `PATH`, which can silently be a different interpreter entirely.

In [ ]:
%pip install -q --upgrade "kaggle-environments>=1.32.6"

In [ ]:
import platform
from importlib.metadata import version

from kaggle_environments import make

print(f"Python              {platform.python_version()}")
print(f"kaggle-environments {version('kaggle-environments')}")

## 2. The simulation

Competition staff have been explicit that **the engine is the source of truth**: the
published documentation and the implementation have disagreed more than once over the
season, and the engine wins every time.

So rather than restating constants from the docs, we read them from the environment
that will actually score us.

In [ ]:
env = make("kaggriculture", debug=False)
config = env.configuration

PARAMETERS_OF_INTEREST = [
    ("episodeSteps", "Turns per season"),
    ("turnsPerDay", "Turns per in-game day"),
    ("boardSize", "Farm width/height in tiles"),
    ("startingMoney", "Opening bank balance"),
    ("shedCapacity", "Non-seed storage cap"),
    ("maxMarketOrdersPerTurn", "Market orders processed per turn"),
    ("actTimeout", "Seconds allowed per turn"),
    ("weedSpawnChance", "Per-tile daily weed chance"),
    ("farmHandCostMult", "Multiplier on the hire cost sequence"),
]

print(f"{'Parameter':26s} {'Value':>10s}   Meaning")
print("-" * 78)
for key, meaning in PARAMETERS_OF_INTEREST:
    print(f"{key:26s} {str(config.get(key)):>10s}   {meaning}")

Two of these constrain agent design far more than the rest.

**`actTimeout` is 1 second per turn**, with a 60-second overage bank for the whole
episode (readable at runtime as `obs["remainingOverageTime"]`). That rules out any
per-turn deep search; the agent must decide with cheap, local reasoning.

**`maxMarketOrdersPerTurn` is 10, and surplus orders are dropped silently** &mdash; no
exception, no warning. An agent that emits twelve orders simply loses two of them.

## 3. Observation schema

Each turn the agent receives a single `obs` dictionary and returns one action per unit
plus a list of market orders.

The critical detail is an axis convention that does not match between two fields:
**`tiles` is row-major (`tiles[y][x]`), while unit positions are `[x, y]`.** Indexing
one with the other produces no error &mdash; just an agent that quietly works the wrong
tile.

In [ ]:
env = make("kaggriculture", configuration={"episodeSteps": 24}, debug=False)
env.run(["starter", "starter"])

observation = env.steps[1][0].observation
farm = observation["farms"][observation["player"]]

print("Top-level observation keys:")
print("  " + ", ".join(sorted(observation.keys())))

print("\nPublic farm state (both players visible):")
for field in ["money", "farmer", "hands", "unlocked_quadrants", "hires_today"]:
    print(f"  {field:20s} {farm[field]}")

print("\nPrivate state (ours only - the opponent's shed is never visible):")
for field, contents in observation["private"].items():
    print(f"  {field:20s} {contents}")

A tile is one of five shapes, and the `kind` key must be checked before assuming any
structure: `None` (empty and unlocked), the string `"LOCKED"`, a plant dictionary, a
weed dictionary, or an animal structure (coop or pasture).

In [ ]:
from collections import Counter


def describe_tiles(farm):
    """Summarise a farm grid by tile kind."""
    kinds = Counter()
    for row in farm["tiles"]:
        for tile in row:
            if tile is None:
                kinds["empty"] += 1
            elif isinstance(tile, str):
                kinds[tile.lower()] += 1
            else:
                kinds[tile.get("kind", "unknown").lower()] += 1
    return kinds


for kind, count in describe_tiles(farm).most_common():
    print(f"  {kind:10s} {count:3d}")

## 4. Agent under test

The agent lives in `main.py` at the repository root, which is also exactly what gets
submitted to Kaggle. The notebook imports that file rather than duplicating strategy
code, so what we measure here is what competes.

One submission rule is worth stating because it fails silently: the framework selects
**the last callable in the module namespace**, not a function named `agent`. A helper
function or class defined below the agent silently becomes the submission, the episode
still reports `DONE`, and the agent finishes on exactly its starting money.

In [ ]:
import pathlib
import subprocess
import sys


def locate_repository():
    """Find main.py locally, or clone the repo when running on Kaggle/Colab."""
    for candidate in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (candidate / "main.py").exists():
            return candidate

    target = pathlib.Path("washamba_bots")
    if not target.exists():
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/Kinjuriu/washamba_bots.git"],
            check=True,
        )
    return target.resolve()


REPO = locate_repository()
AGENT_PATH = str(REPO / "main.py")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import main

print(f"Repository: {REPO}")
print(f"Submitted entry point resolves to: {main.agent.__name__}")

## 5. Evaluation methodology

**A single episode cannot distinguish an improvement from luck.** Run-to-run spread on
an identical agent has exceeded 1,400 bank on the same matchup, which is larger than
most changes we would like to detect.

Two rules follow, and both are load-bearing:

1. **Always evaluate over a batch of fixed seeds**, and report the mean and win rate
   rather than any individual score.
2. **A/B strategy changes against `pass` and `starter`.** Passing a `seed` makes the
   *environment* deterministic &mdash; weed spawns and shop unlocks &mdash; but it does
   not control the built-in `random` agent's own RNG, so that opponent still drifts
   between runs on an identical seed.

In [ ]:
import statistics


def evaluate(agent_path, opponent, seeds, episode_steps=720):
    """Play one seeded episode per seed and collect the closing balances."""
    rows = []
    for seed in seeds:
        env = make(
            "kaggriculture",
            configuration={"episodeSteps": episode_steps, "seed": seed},
            debug=False,
        )
        env.run([agent_path, opponent])
        ours, theirs = env.steps[-1]
        rows.append(
            {
                "opponent": opponent,
                "seed": seed,
                "ours": ours.reward,
                "theirs": theirs.reward,
                "won": ours.reward > theirs.reward,
            }
        )
    return rows


def summarise(rows):
    """Reduce per-episode rows to the numbers worth reporting."""
    ours = [row["ours"] for row in rows]
    return {
        "episodes": len(rows),
        "mean": round(statistics.mean(ours)),
        "stdev": round(statistics.stdev(ours)) if len(ours) > 1 else 0,
        "min": round(min(ours)),
        "max": round(max(ours)),
        "wins": sum(row["won"] for row in rows),
    }

### Running the batch

Each 720-turn episode takes roughly seven seconds. `SEEDS` below is deliberately small
so the notebook stays responsive; **use twelve or more seeds before acting on a
result.** The three built-in opponents are `pass` (does nothing, banks its opening
3,000), `random`, and `starter` (a deterministic reference agent).

In [ ]:
SEEDS = range(4)
OPPONENTS = ["pass", "random", "starter"]

per_seed = {}
results = {}
for opponent in OPPONENTS:
    per_seed[opponent] = evaluate(AGENT_PATH, opponent, SEEDS)
    results[opponent] = summarise(per_seed[opponent])
    print(f"  {opponent:8s} complete")

## 6. Results

### Visual conventions

Every figure below draws from one small validated palette, applied in a fixed order
so a given colour always means the same thing across charts. The three hues clear
colourblind-separation and normal-vision thresholds as a set. Grid lines and axes stay
recessive so the data carries the emphasis, and no chart uses two y-axes &mdash; where
two measures have different units they get separate figures.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "#e5e4e0"

# Fixed categorical order - assigned by position, never cycled.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK_MUTED,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "font.size": 10,
    "figure.dpi": 110,
})


def finish(ax, title, xlabel=None, ylabel=None, legend=False):
    """Apply the shared title/label/legend treatment to an axis."""
    ax.set_title(title, fontsize=12.5, color=INK, pad=12, loc="left")
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if legend:
        ax.legend(frameon=False, labelcolor=INK_MUTED)
    plt.tight_layout()


STARTING_MONEY = config.get("startingMoney", 3000)

summary = pd.DataFrame(results).T
summary.index.name = "opponent"
summary

### Distribution across seeds

A bar of means would hide the thing that matters most here: **spread**. Each dot is one
seeded episode, so the vertical scatter within a column is exactly the run-to-run noise
that makes single-episode comparisons untrustworthy.

The dashed line is the opening balance. It is the floor that counts &mdash; an agent
finishing below it has destroyed value relative to doing nothing at all, which an
earlier revision of this agent genuinely did on unlucky seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))

for index, opponent in enumerate(OPPONENTS):
    balances = [row["ours"] for row in per_seed[opponent]]
    offsets = [
        index + (position - (len(balances) - 1) / 2) * 0.07
        for position in range(len(balances))
    ]
    ax.scatter(
        offsets, balances,
        s=48, color=SERIES[index], zorder=3,
        edgecolor=SURFACE, linewidth=1.5,
    )

    mean_balance = statistics.mean(balances)
    ax.hlines(mean_balance, index - 0.25, index + 0.25,
              color=INK, linewidth=2, zorder=4)
    ax.text(index + 0.29, mean_balance, f"mean {mean_balance:,.0f}",
            va="center", fontsize=9, color=INK)

ax.axhline(STARTING_MONEY, color=INK_MUTED, linestyle="--", linewidth=1, zorder=1)
ax.text(-0.42, STARTING_MONEY, f"opening balance {STARTING_MONEY:,}",
        va="bottom", fontsize=8.5, color=INK_MUTED)

ax.set_xticks(range(len(OPPONENTS)))
ax.set_xticklabels(OPPONENTS)
ax.set_xlim(-0.5, len(OPPONENTS) - 0.25)

finish(ax, "Closing balance per seeded episode",
       xlabel="Opponent", ylabel="Bank balance")
plt.show()

## 7. Behavioural diagnostics

Aggregate scores say whether a change helped. They do not say why. For that we replay a
single episode and record how it unfolded: the balance over time, the state of the
farm, and where the turns went.

One episode is the right unit here &mdash; we are reading mechanism, not measuring
performance.

In [ ]:
def profile_episode(agent_path, opponent, seed):
    """Replay one episode, recording per-day history and end state."""
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": seed},
        debug=False,
    )
    env.run([agent_path, opponent])

    turns_per_day = config.get("turnsPerDay", 24)
    unit_actions = Counter()
    market_orders = Counter()
    history = []

    for index, step in enumerate(env.steps):
        action = step[0].get("action") or {}
        farmer = action.get("farmer") or []
        if farmer:
            unit_actions[farmer[0]] += 1
        for hand in action.get("hands") or []:
            if hand:
                unit_actions[hand[0]] += 1
        for order in action.get("market") or []:
            if order:
                market_orders[order[0]] += 1

        # Sample once per in-game day to keep the series readable.
        if index % turns_per_day == 0:
            observation = step[0].observation
            tiles = describe_tiles(observation["farms"][0])
            history.append({
                "day": index // turns_per_day,
                "ours": observation["farms"][0]["money"],
                "theirs": observation["farms"][1]["money"],
                "plants": tiles.get("plant", 0),
                "weeds": tiles.get("weed", 0),
                "empty": tiles.get("empty", 0),
                **{f"price_{k}": v for k, v in observation["market"]["prices"].items()},
            })

    final = env.steps[-1][0]
    return {
        "balance": final.reward,
        "unit_actions": unit_actions,
        "market_orders": market_orders,
        "final_farm": final.observation["farms"][0],
        "history": pd.DataFrame(history).set_index("day"),
    }


profile = profile_episode(AGENT_PATH, "starter", seed=0)
timeline = profile["history"]

print(f"Closing balance: {profile['balance']:,.0f}\n")
print("Market orders issued:")
for order, count in profile["market_orders"].most_common():
    print(f"  {order:12s} {count:4d}")

print("\nFarm at season end:")
for kind, count in describe_tiles(profile["final_farm"]).most_common():
    print(f"  {kind:12s} {count:4d}")

### Where the season is won or lost

Closing balance is a single number at turn 720; this is the path it took. Divergence
that opens early and never closes points at a structural problem, while a flat stretch
means the agent stopped converting harvests into cash.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))

ax.plot(timeline.index, timeline["ours"], color=SERIES[0],
        linewidth=2, label="nikaangukia_meroni", zorder=3)
ax.plot(timeline.index, timeline["theirs"], color=SERIES[1],
        linewidth=2, label="starter", zorder=3)

ax.axhline(STARTING_MONEY, color=INK_MUTED, linestyle="--", linewidth=1, zorder=1)

# Direct labels at the line ends, so identity never rests on colour alone.
for column, colour, name in [("ours", SERIES[0], "ours"),
                             ("theirs", SERIES[1], "starter")]:
    ax.text(timeline.index[-1] + 0.4, timeline[column].iloc[-1],
            f"{name} {timeline[column].iloc[-1]:,.0f}",
            va="center", fontsize=9, color=INK)

ax.set_xlim(0, timeline.index[-1] + 6)
finish(ax, "Bank balance across the season",
       xlabel="Day", ylabel="Bank balance", legend=True)
plt.show()

### Farm health over time

This is the chart that explains the balance curve. Tiles are a fixed budget, so the
question is what fraction is productive on any given day.

**Weeds are the signal to watch.** A weeded tile is dead ground until it is dug back,
and a rising weed band means the crew is planting faster than it can maintain &mdash;
every tile in that band is capital already spent and now earning nothing.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))

ax.stackplot(
    timeline.index,
    timeline["plants"], timeline["weeds"], timeline["empty"],
    labels=["Live plants", "Weeds", "Empty"],
    colors=SERIES,
    edgecolor=SURFACE,
    linewidth=1.5,
)

ax.set_xlim(timeline.index[0], timeline.index[-1])
ax.set_ylim(0, None)
finish(ax, "Farm composition across the season (owned tiles)",
       xlabel="Day", ylabel="Tiles", legend=True)
plt.show()

### Where the turns go

Every unit gets exactly one action per turn, so the season is a fixed budget of turns
split across the farmer and any hired hands. Movement is the overhead line: turns spent
walking are turns not spent watering, digging, or harvesting.

In [ ]:
actions = profile["unit_actions"].most_common(10)
labels = [name for name, _ in actions][::-1]
values = [count for _, count in actions][::-1]

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.barh(labels, values, color=SERIES[0], edgecolor=SURFACE, linewidth=1.5, zorder=3)
ax.xaxis.grid(True)
ax.yaxis.grid(False)

for position, value in enumerate(values):
    ax.text(value + max(values) * 0.01, position, f"{value:,}",
            va="center", fontsize=9, color=INK_MUTED)

ax.set_xlim(0, max(values) * 1.12)
finish(ax, "Turns by action type (farmer and hired hands)", xlabel="Turns")
plt.show()

### Market response to our selling

Sale prices move with market inventory, and both players' orders clear one unit at a
time against the same book. Premium goods fall toward the floor fast on oversupply, so
a price line that sags after our harvests is direct evidence that we are selling too
much at once and should spread those orders out.

In [ ]:
TRACKED_PRODUCTS = ["WHEAT", "CARROT", "TOMATO"]
price_columns = [f"price_{product}" for product in TRACKED_PRODUCTS]
available = [column for column in price_columns if column in timeline.columns]

fig, ax = plt.subplots(figsize=(8.5, 4.6))

for index, column in enumerate(available):
    product = column.removeprefix("price_")
    ax.plot(timeline.index, timeline[column], color=SERIES[index],
            linewidth=2, label=product.title(), zorder=3)
    ax.text(timeline.index[-1] + 0.4, timeline[column].iloc[-1],
            f"{product.title()} {timeline[column].iloc[-1]:,.0f}",
            va="center", fontsize=9, color=INK)

ax.set_xlim(0, timeline.index[-1] + 7)
finish(ax, "Market sale price across the season",
       xlabel="Day", ylabel="Price per unit", legend=True)
plt.show()

## 8. Findings

Three measurements changed how the agent is built. Each was found through the
diagnostics above rather than by reasoning about the rules.

**Tile upkeep capacity gates income, not sell prices.** An early revision planted more
tiles than one farmer could water. Plants weeded out, and because the agent never
issued `DIG`, each weeded tile stayed dead for the rest of the season. The farm decayed
to 23 of 25 tiles dead and sales starved to under four `SELL` orders per season. Adding
`DIG` moved every metric at once. Before optimising thresholds, check how many tiles
are still alive at season end.

**Hiring is the highest-return mechanic available.** The n-th hire of a day costs
`farmHandCostMult * fib(n)` and the counter resets each morning, so four hands cost
1 + 1 + 2 + 3 = 7 per day, roughly 210 for a full season. That expenditure is worth
well over a thousand in closing balance, because extra units raise exactly the upkeep
ceiling identified above. Hands are cleared nightly and must be re-hired each morning.

**Multiple units need explicit coordination.** Every unit running the same
"nearest useful tile" rule independently sends the whole crew to a single tile. Units
must reserve their targets so the work spreads across the farm.

## 9. Next steps

Ordered by expected value:

1. **`BUY_LAND`.** The crew can now maintain more tiles than the opening quadrant
   provides. Quadrants cost 1,000 / 2,000 / 4,000, so the question is when the marginal
   tile repays that outlay within a 30-day season.
2. **Animals.** Geese, cows, and sheep yield indefinitely while fed and never decay
   into weeds, but they require a wheat supply and daily feeding. This needs a
   grow-your-own-feed subsystem to be worth it.
3. **`FERTILIZE`.** Doubles the yield bonus for three days, but only on days the plant
   is also watered. Highest return on high-value, reliably tended plants.
4. **Sell scheduling.** Premium goods collapse toward the price floor on oversupply,
   and both players' orders clear one unit at a time against a shared market. Spreading
   sales should beat dumping a harvest in one order.